In [1]:
!pip install -r requirements.txt

In [2]:
!pip install langchain-nvidia-ai-endpoints

In [3]:
with open("./Moby-Dick.txt", "r", encoding='utf-8') as f:
    book= f.read()

In [4]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_text_splitters import TokenTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
import getpass


In [5]:
# NVIDIA_API_KEY= "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"

In [6]:
llm = ChatNVIDIA(
  model="openai/gpt-oss-20b",
  api_key="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", 
  temperature=1,
  top_p=1,
)


## Split

In [28]:
text_chunks_chain= (
    RunnableLambda(lambda x:
                   [
                       {
                           'chunk': text_chunks,
                       }
                       for text_chunks in TokenTextSplitter(chunk_size=3000, chunk_overlap=100).split_text(x)
                   ]
    )
)

## Map

In [29]:
summarize_CHUNK_prompt_TEMPLATE = """
    Viết lại vắn tắt những dòng text dưới đây, gồm cả những chi tiết chính
    Text: {chunk}
"""
summarize_CHUNK_prompt = PromptTemplate.from_template(summarize_CHUNK_prompt_TEMPLATE)
summarize_CHUNK_CHAIN = summarize_CHUNK_prompt | llm

summarize_MAP_CHAIN = (
    RunnableParallel( 
        # Nhận 1 chunk → chạy summarize_CHUNK_CHAIN
        # → StrOutputParser() → tạo {"summary": "..."}
        {
            'summary' : summarize_CHUNK_CHAIN | StrOutputParser()
            
            
            
        }
    )

)

## Reduce

In [33]:
sum_sums_from_map_prompt_tem = """
Write a concise summary of the following text, which joins several summaries, and include the main details. 
Text: {summaries}
"""

sum_sums_from_map_prompt = PromptTemplate.from_template(sum_sums_from_map_prompt_tem)
sum_reduce_chain = (
    RunnableLambda(lambda x:
                   {
                       'summaries' : '\n'.join([i['summary'] for i in x]),
                   }
    ) | sum_sums_from_map_prompt | llm | StrOutputParser()
)

## MapReduce

In [34]:
map_reduce_chain = (
    text_chunks_chain | summarize_MAP_CHAIN.map() | sum_reduce_chain
)

In [35]:
tomtat = map_reduce_chain.invoke(book)

In [37]:
print(tomtat)

**Tóm tắt ngắn gọn – “Moby‑Dick” (đề: “Chí” của Tôm bi”)**

| Mục tiêu | Nội dung chính |
|---|---|
| **Thông tin bản quyền** | • Tác giả: Herman Melville  <br>• Ebook tự do (US‑based / thu nhập < K²) 27/06/2001  <br>• Ngôn ngữ: Tiếng Anh (UTF‑8) |
| **Giới thiệu tổng quan** | – Đoạn mở đầu, “CHAPTER 1. Loomings.”  <br>– Ishmael “Call me Ishmael” tạo hình thủy thủ tự bộ, muốn dội vào biển để trốn thoát gánh nặng, tìm “vàng xưa của tự do”. |
| **Những điểm nổi bật đầu chapter** | 1. **Mơ ám của Ishmael** – khao khát thoát khỏi giới hạn cuộc sống thành phố.  <br>2. **Mô tả Manhattan** – giai điệu bèo chợ, cảnh biển, “đêm vũ trụ”.  <br>3. **Những gì biển gợi nhớ** – 呼吸, hơi thở mạnh mẽ, “đưa mọi người ra” <br>4. **Thật là một nhân vật lạc lõng** – không quan tâm tới danh vọng, chỉ muốn sống dưới “cờ gió”. |
| **CHAPTER 2 – “The Carpet‑Bag”** | - Ishmael đóng đồ vào lọ vải, rời Manhattan, chuẩn bị đón hành trình.  <br>- Vị trí: New Bedford, bắt đầu tham gia thủy thủ ngắn hạn.  <br>- “Nantu

# Tóm tắt các tài liệu (mọi loại định dạng)

In [10]:
import wikipedia

wikipedia.set_user_agent(
    "MyLangChainApp/1.0 (your-email@example.com)"
)

In [14]:
from langchain_community.document_loaders import WikipediaLoader

wikipedia_loader= WikipediaLoader(query= "Ronaldo", load_max_docs=2)
wikipedia_docs = wikipedia_loader.load()

In [15]:
print(wikipedia_docs)

[Document(metadata={'title': 'Cristiano Ronaldo', 'summary': "Cristiano Ronaldo dos Santos Aveiro (born 5 February 1985) is a Portuguese professional footballer who plays as a forward for and captains the Saudi Pro League club Al-Nassr and the Portugal national team. Nicknamed CR7, he is widely regarded as one of the greatest players in history, having won numerous individual accolades throughout his career including five Ballon d'Or awards, a record three UEFA Men's Player of the Year Awards, and four European Golden Shoes. He was named the world's best player five times by FIFA.\nRonaldo is one of the most decorated players in the history of professional football, having won 35 trophies in his career, including five UEFA Champions Leagues, two UEFA Nations Leagues and the UEFA European Championship. He holds the records for most goals (140) and assists (42) in the Champions League, most goals (14) and assists (8) in the European Championship, most international appearances (233), mos

In [52]:
!pip install python-docx

In [55]:
from docx import Document
doc = Document()

doc.add_heading("Wikipedia - Messi", level=1)

for i, wikipedia_doc in enumerate(wikipedia_docs, start=1):
    doc.add_heading(f"Tài liệu {i}", level=2)

    doc.add_paragraph(
        f"Metadata: {wikipedia_doc.metadata}"
    )

doc.save("tailieu_messi.docx")

print("tao file thanh cong")

tao file thanh cong


In [19]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader, TextLoader
word_doc = Docx2txtLoader("tailieu_messi.docx").load()
pdf_doc = PyPDFLoader("messi.pdf").load()
all_docs = word_doc+ pdf_doc + wikipedia_docs

In [20]:
doc_sum_prompt_template = """
    Hãy tóm tắt lại cái doc này
    File: {file}
"""
doc_sum_prompt = PromptTemplate.from_template(doc_sum_prompt_template)
doc_sum_chain = doc_sum_prompt | llm

 Thiết lập chuỗi để tinh chỉnh bản tóm tắt bằng cách kết hợp lặp lại bản tóm tắt hiện tại với bản tóm tắt của tài liệu bổ sung

In [21]:
refine_sum_template ="""
Bạn cần phải đưa ra bản tóm tắt cuối cùng từ refine summary hiện tại cái mà được sinh ra từ trước gồm nội dung của nhiều tài liệu được
gọp lại 
Đây là bản refine summary trước đó {current_refined_summary}
Còn đây là nội dung được bổ sung thêm vào: {text}
Chỉ sử dụng nội dung được thêm vào khi thấy nó có ích, còn không thì trả về đầy đủ summary hiện tại 
"""
refine_sum_prompt = PromptTemplate.from_template(refine_sum_template)
refine_sum_chain = refine_sum_prompt | llm | StrOutputParser()


Cuối cùng thì viết hàm lặp qua từng tài liệu, tóm tắt nó bằng doc_summary_chain và tinh chỉnh tóm tắt tổng thể bằng refine_chain

In [25]:
def refine_sum(docs):
    intermediate_steps = []
    current_refined_summary =''
    for doc in docs:
        intermediate_step = {
            "current_refined_summary": current_refined_summary,
            "text": doc.page_content
        }
        intermediate_steps.append(intermediate_step)
        current_refined_summary = refine_sum_chain.invoke(intermediate_step)

    return {
        "final_sum": current_refined_summary,
        "intermediate_steps": intermediate_steps
    }

In [26]:
full_summary = refine_sum(all_docs)

In [27]:
print(full_summary)

{'final_sum': "**BẢN TÓM TẮT CHI TIẾT TỪ “Refine Summary” – LIONEL “LEO” MESSI & CRISTIANO RONALDO**\n\n| Thông tin | Lionel\u202fMessi | Cristiano\u202fRonaldo |\n|-----------|--------------|--------------------|\n| **Tên đầy đủ** | Lionel\u202fAndrés\u202fMessi | Cristiano\u202fRonaldo\u202fdos\u202fSantos\u202fAveiro |\n| **Ngày sinh** | 24/06/1987 | 05/02/1985 |\n| **Quốc tịch** | Argentina | Bồ Đào Nha |\n| **Vị trí** | Mặt tiền / Đột phòng | Mặt tiền |\n| **Cơ địa** | 1.70\u202fm / 66\u202fkg | 1.87\u202fm / 83\u202fkg |\n| **Cập nhật hiện tại** | Đúng hợp đồng với Inter\u202fMiami (MLS) | Đang huấn luyện và trưởng nhóm Al‑Nassr (Saudi Pro League) |\n| **Các club chủ đạo** | Inter\u202fMiami (từ 2023), trước đó: PSG (2021‑2023), Barcelona (2004‑2021) | Al‑Nassr (từ 2023), trước đó: Juventus (2018‑2021), Real\u202fMadrid (2009‑2018), Manchester\u202fUnited (2003‑2009\u202f; 2021‑2022) |\n| **Cấp độ quốc gia** | Argentina – Quý cựu trưởng đội (thời gian 2011‑2020) | Bồ Đào Nha – Ti